# P12 · Alpha, Beta y factores financieros
**QuantLab | Módulo 1 · Fundamentos financieros | Práctica 12**

Pregunta: ¿la estrategia gana por exposición al mercado, a factores conocidos o por un componente que el modelo no explica?

Aprenderás a estimar una regresión de Jensen, extenderla a varios factores, atribuir rentabilidad, contrastar el intercepto, congelar las exposiciones para evaluar fuera de muestra y construir un factor long-short sencillo.

**Ruta de trabajo:** observa los ejemplos resueltos, ejecuta cada celda, revisa los ejercicios resueltos de la sección 10 y repite el análisis con los retornos diarios netos exportados de P09. Los resultados iniciales son sintéticos, no una recomendación de inversión.

Las funciones están en **`src/quantlab/p12/factors.py`** y sus pruebas en **`tests/test_factors.py`** (`pytest` desde la raíz del proyecto).

## 1. Configuración
Usa Colab, Jupyter o VS Code. Si faltan paquetes, ejecuta `%pip install numpy pandas matplotlib statsmodels` en una celda. Los retornos son decimales simples por día, no porcentajes. Se utiliza un calendario de 365 días para cripto. En otros mercados hay que cambiar tanto calendario como anualización.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
from IPython.display import display
import inspect
import json
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p12.factors import (active_vs_alpha, annualized_attribution, attribution, chronological_split,
                                  daily_risk_free, daily_rule_returns, factor_after_costs, fit_model,
                                  frozen_validation, long_short_factor, manual_alpha_beta, model_table,
                                  omitted_factor_table, read_daily, rolling_beta, synthetic_factor_data,
                                  synthetic_universe, validate_daily_factors)

DATA_MODE = 'synthetic'  # 'synthetic' o 'csv'
RETURNS_CSV = 'daily_returns.csv'  # formato P09: timestamp, net_return, benchmark_return
FACTORS_CSV = None  # opcional: timestamp, momentum_factor, liquidity_factor, risk_free_return
RISK_FREE_ANNUAL = 0.02  # supuesto, no cotización real
PERIODS_PER_YEAR = 365
TRAIN_FRACTION = 0.65
HAC_LAGS = 5
ROLLING_WINDOW = 90
SEED = 42
EXPORT = False
plt.rcParams.update({'figure.figsize':(9,3.6),'figure.dpi':110,'axes.grid':True,
                     'grid.alpha':.2,'axes.spines.top':False,'axes.spines.right':False})
assert 0 < TRAIN_FRACTION < 1 and PERIODS_PER_YEAR > 0 and RISK_FREE_ANNUAL > -1
assert HAC_LAGS >= 0 and ROLLING_WINDOW >= 2
rf_daily = daily_risk_free(RISK_FREE_ANNUAL, PERIODS_PER_YEAR)

## 2. Del rendimiento absoluto al rendimiento excedente
Si $r_s$ es el retorno de la estrategia, $r_m$ el del benchmark y $r_f$ el tipo libre de riesgo del mismo periodo:

- Rentabilidad activa: $r_s-r_m$.
- Rentabilidad excedente: $r_s-r_f$.
- Modelo de un factor (Jensen): $r_s-r_f=\alpha+\beta(r_m-r_f)+\varepsilon$.

Una beta de 1,5 indica que, según el modelo, una variación de 1 punto porcentual del exceso de retorno del mercado se asocia con 1,5 puntos de variación en el exceso de retorno de la estrategia. No es una garantía ni una relación causal.

El alpha es un intercepto medio por periodo, condicionado al modelo. No es la diferencia simple frente al benchmark. Un alpha positivo puede proceder de factores omitidos, errores de medición o azar.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿los retornos diarios son utilizables?</b>

`validate_daily_factors` se detiene con un error si:

- Hay valores ausentes o no finitos: **no se imputan** retornos sin investigar.
- Falta algún día: no se rellenan huecos con cero.
- Hay menos de 100 días, o algún retorno ≤ −100 %.

Después añade `market_excess` = r_m − r_f. `chronological_split` exige que desarrollo y validación tengan más observaciones que los retardos HAC y los parámetros del modelo.

</div>

In [ ]:
print(inspect.getsource(synthetic_factor_data))
print(inspect.getsource(validate_daily_factors))

if DATA_MODE == 'synthetic':
    daily = synthetic_factor_data(SEED, 900, rf_daily)
    strategies = ['A','B']
    provenance = {'mode':DATA_MODE,'seed':SEED,'description':'retornos netos sintéticos, sin alpha estructural en A; 4.5 bps diarios en B'}
elif DATA_MODE == 'csv':
    daily = read_daily(RETURNS_CSV).rename(columns={'net_return':'strategy'})
    if not {'strategy','benchmark_return'}.issubset(daily):
        raise ValueError('CSV P09: requiere net_return y benchmark_return')
    daily = daily[['strategy','benchmark_return']].copy()
    if FACTORS_CSV:
        factors = read_daily(FACTORS_CSV)
        allowed = ['momentum_factor','liquidity_factor','risk_free_return']
        daily = daily.join(factors[[c for c in allowed if c in factors]],how='left')
    if 'risk_free_return' not in daily: daily['risk_free_return'] = rf_daily
    strategies=['strategy']
    provenance={'mode':DATA_MODE,'returns_csv':RETURNS_CSV,'factors_csv':FACTORS_CSV}
else: raise ValueError('DATA_MODE desconocido')

daily = validate_daily_factors(daily, strategies)
factor_cols=['market_excess']+[c for c in ['momentum_factor','liquidity_factor'] if c in daily]
train, test = chronological_split(daily, TRAIN_FRACTION, HAC_LAGS, len(factor_cols))
print(provenance, len(train), 'días desarrollo |', len(test), 'días validación')
display(daily.head().round(6))

### 3. Benchmark y curva de capital
Compararemos la capitalización de 100 € hipotéticos con el mismo horizonte. El benchmark debe compartir moneda, frecuencia y convención de costes. Si el benchmark importado es BTC bruto y la estrategia es neta, documenta que la comparación no descuenta los costes del benchmark.

In [ ]:
wealth=(1+daily[['benchmark_return']+strategies]).cumprod()*100
wealth.loc[daily.index[0]-pd.Timedelta(days=1)]=100
wealth=wealth.sort_index()
ax=wealth.plot(title=f'Capital acumulado (base 100) · {DATA_MODE}')
ax.axvline(test.index[0],color='grey',ls='--',label='Inicio validación')
ax.set_ylabel('Capital (base 100)'); ax.legend(); plt.tight_layout(); plt.show()
display((wealth.iloc[-1]/100-1).mul(100).rename('retorno_total_pct').to_frame().round(2))

## 4. Estimación de Alpha y Beta
OLS minimiza la suma de los residuos al cuadrado. Para un único factor, $\beta=\operatorname{Cov}(r_m-r_f,r_s-r_f)/\operatorname{Var}(r_m-r_f)$ y $\alpha=\overline{r_s-r_f}-\beta\overline{r_m-r_f}$.

Usamos errores estándar HAC (Newey-West) como introducción a la inferencia con heterocedasticidad y autocorrelación. No solucionan selección de modelos, factores omitidos ni muestras pequeñas. El p-valor contrasta alpha cero, no la probabilidad de que la estrategia sea rentable.

In [ ]:
print(inspect.getsource(fit_model))
print(inspect.getsource(model_table))

specs={'Mercado':['market_excess']}
if len(factor_cols)>1: specs['Multifactor']=factor_cols
model_results, models = model_table(train, strategies, specs, HAC_LAGS, PERIODS_PER_YEAR)
display(model_results.round(4))

El alpha anualizado anterior es una extrapolación aritmética, no CAGR ni rentabilidad compuesta garantizada. Compara el intercepto de A antes y después de introducir los demás factores; un intercepto puede cambiar aunque las operaciones no hayan cambiado.

In [ ]:
fig,ax=plt.subplots()
x=np.arange(len(model_results))
y=model_results.alpha_daily_bps
ax.errorbar(x,y,yerr=[y-model_results.ci_low_bps,model_results.ci_high_bps-y],fmt='o',capsize=4)
ax.axhline(0,color='grey',ls='--'); ax.set_xticks(x,model_results.strategy+' · '+model_results.model)
ax.set_ylabel('Alpha diario (bps)'); ax.set_title('Alpha en desarrollo e IC del 95 % (HAC)')
plt.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · El alpha de A era exposición a factores; el de B, no</b>

| Estrategia | Modelo | Alpha diario | IC 95 % | p-valor | Beta | R² |
|---|---|---|---|---|---|---|
| A | Mercado | **6,29 bps** | [0,44; 12,15] | **0,035** | 0,84 | 0,83 |
| A | Multifactor | 1,09 bps | [−1,23; 3,42] | 0,356 | 0,75 | 0,97 |
| B | Mercado | 10,79 bps | [4,94; 16,65] | 0,0003 | 0,84 | 0,83 |
| B | Multifactor | **5,59 bps** | [3,27; 7,92] | < 0,0001 | 0,75 | 0,97 |

- **A no tiene alpha** por construcción. Con el modelo de mercado aparece un alpha "significativo" de 6,29 bps diarios (23 % anual aritmético). Al añadir momentum y liquidez, cae a 1,09 bps y deja de ser significativo: era **exposición a factores que el modelo de un factor no veía**.
- **B tiene 4,5 bps diarios** de alpha programado. El modelo multifactor estima 5,59 bps, con un IC que contiene el valor verdadero, y sigue siendo significativo.
- El R² sube de 0,83 a 0,97: los factores explican mucho más. Pero un R² alto no prueba que el modelo sea correcto (ver E3).

</div>

## 5. Atribución: de dónde sale cada punto básico
Para un modelo multifactor: $r_s=r_f+\alpha+\sum_j\beta_j f_j+\varepsilon$. Sumaremos las medias de cada término, expresadas en bps diarios. La suma es exacta para rentabilidades aritméticas, no para rentabilidad compuesta.

Los factores long-short de este ejemplo ya son diferenciales de retorno: no se les vuelve a restar el tipo libre de riesgo. En los datos reales debe conocerse su metodología, universo y moneda.

In [ ]:
print(inspect.getsource(attribution))

chosen=list(specs)[-1]
attr_train=pd.DataFrame({s:attribution(train,s,models[s,chosen]).mean()*1e4 for s in strategies}).T
display(attr_train.round(3))
ax=attr_train.plot.bar(rot=0,title=f'Atribución media diaria en desarrollo · {chosen}')
ax.set_ylabel('bps diarios'); ax.legend(fontsize=8,ncol=2);plt.tight_layout();plt.show()

## 6. Validación cronológica: congelar las exposiciones
No recalibramos betas ni alpha para mejorar el periodo reservado. Calculamos el retorno atribuido usando parámetros de desarrollo y factores realizados en validación. El error medio fuera de muestra es el retorno no explicado por el modelo congelado, **no una predicción anterior al cierre**. Los factores contemporáneos no se conocen anticipadamente.

También estimamos una regresión separada en validación exclusivamente como diagnóstico de estabilidad, no para decidir retroactivamente operaciones. El contraste es exploratorio; para una prueba final se necesita otro periodo no consultado.

In [ ]:
print(inspect.getsource(frozen_validation))

validation, out_parts = frozen_validation(test, strategies, models, chosen, specs[chosen], HAC_LAGS, PERIODS_PER_YEAR)
display(validation.round(4))
ax=pd.DataFrame({s:out_parts[s].residual.cumsum()*1e4 for s in strategies}).plot(title='Residuos acumulados fuera de muestra (suma aritmética)')
ax.set_ylabel('bps acumulados');plt.tight_layout();plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Fuera de muestra: el alpha de B se mantiene, pero B pierde dinero</b>

| | A | B |
|---|---|---|
| Retorno en validación | **−27,2 %** | **−16,1 %** |
| Benchmark | −26,8 % | −26,8 % |
| No explicado por el modelo congelado | 1,28 bps/día | 5,78 bps/día |
| Alpha diagnóstico en validación | 1,00 bps (p = 0,57) | **5,50 bps (p = 0,002)** |
| Information ratio | −0,11 | 1,08 |

- El alpha de B es **estable**: 5,59 bps en desarrollo y 5,50 en la regresión diagnóstica de validación. El de A sigue sin ser significativo.
- Aun así, **las dos pierden capital** en validación, porque el mercado cayó un 27 % y tienen beta 0,75. Un alpha positivo no protege de la exposición al mercado.
- B termina un **15 %** por encima de A ((1 − 0,161) / (1 − 0,272) − 1): aproximadamente lo que acumulan 315 días de 4,5 bps de alpha (1,00045³¹⁵ − 1 ≈ 15 %).

</div>

## 7. Beta móvil: ¿la exposición es constante?
La beta móvil de un factor se calcula con las últimas 90 observaciones conocidas en cada fecha. No es exactamente la beta parcial de una regresión multifactor. Si se utiliza para una orden, debe estar disponible antes de la ejecución.

En el ejemplo sintético las curvas de beta y residuos de A y B se superponen porque B solo añade una constante a A; esa diferencia se recoge en el intercepto.

In [ ]:
print(inspect.getsource(rolling_beta))

rolling = rolling_beta(daily, strategies, ROLLING_WINDOW)
ax=rolling.plot(title=f'Beta móvil unifactorial · {ROLLING_WINDOW} días')
ax.axvline(test.index[0],color='grey',ls='--');ax.set_ylabel('Beta');plt.tight_layout();plt.show()

## 8. Construir un factor Momentum long-short desde cero
Usaremos seis activos ficticios. En el cierre de ayer se ordenan por su retorno acumulado en 20 días; hoy se mide el retorno de los dos mejores menos el de los dos peores. No se usa el retorno de hoy para elegir los activos de hoy. El resultado es un diferencial de retornos, no un rendimiento de capital de una cartera financiada; faltan costes, disponibilidad de cortos y restricciones de ejecución.

Este factor de juguete es independiente del factor artificial de la sección 2. No lo mezclamos con la estrategia del CSV para inventar una atribución real.

In [ ]:
print(inspect.getsource(long_short_factor))

universe = synthetic_universe(SEED+1)
momentum_ls = long_short_factor(universe, 20, 2)
score, long_mask, short_mask, factor = momentum_ls['score'], momentum_ls['long_mask'], momentum_ls['short_mask'], momentum_ls['factor']
display(pd.concat([score.tail(3).add_prefix('score_'),factor.rename('constructed_momentum').tail(3)],axis=1).round(4))
print('Primer retorno disponible:',factor.first_valid_index())

## 9. Comprobaciones matemáticas
Utilizamos una serie pequeña conocida y reconciliamos los componentes del modelo. Las pruebas verifican cálculos y alineación, no la viabilidad económica de las estrategias.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: parámetros conocidos, separación temporal, reconciliación y factor sin futuro
toy_x=np.linspace(-.02,.02,100)
toy=pd.DataFrame({'market_excess':toy_x,'risk_free_return':0.,'toy':.001+1.5*toy_x})
toy_fit=fit_model(toy,'toy',['market_excess'])
assert np.isclose(toy_fit.params['const'],.001)
assert np.isclose(toy_fit.params['market_excess'],1.5)
assert train.index.max()<test.index.min()
assert len(train)+len(test)==len(daily)
assert np.isclose((1+rf_daily)**PERIODS_PER_YEAR-1,RISK_FREE_ANNUAL)
for s in strategies:
    for frame in [train,test]:
        parts=attribution(frame,s,models[s,chosen])
        assert np.allclose(parts.sum(axis=1),frame[s])
    assert abs(attribution(train,s,models[s,chosen]).residual.mean())<1e-10
assert factor.iloc[:20].isna().all()
assert np.isfinite(factor.iloc[20:]).all()
assert np.allclose(score.iloc[20].to_numpy(),(1+universe.iloc[:20]).prod().to_numpy()-1)
assert (long_mask.iloc[20:].sum(axis=1)==2).all() and (short_mask.iloc[20:].sum(axis=1)==2).all()
print('Comprobaciones matemáticas y temporales superadas.')

## 10. Ejercicios
**E1 — Beta manual.** Implementa covarianza/varianza y alpha a partir de las medias. Compara con OLS. Pista: usa retornos excedentes, no precios.

**E2 — Benchmark.** Compara A y B con BTC sintético, y explica por qué retorno activo y alpha pueden diferir.

**E3 — Factores omitidos.** Elimina un factor de la regresión; registra cómo cambian alpha, beta, IC y R². ¿Un R² mayor prueba que el modelo es correcto?

**E4 — Atribución.** A partir de `attr_train`, expresa los componentes en porcentaje anual aritmético y reconcilia la suma. Distingue error residual e intercepto.

**E5 — Estabilidad.** Cambia la ventana móvil a 30 y 180 días y compara sensibilidad. Reserva un nuevo periodo antes de extraer conclusiones si ajustas parámetros.

**E6 — Construcción de factores.** Construye un factor de reversión invirtiendo los grupos del ejemplo. Verifica la fecha de disponibilidad y analiza costes hipotéticos.

**E7 — Datos propios.** Exporta `daily_returns.csv` desde P09 (`EXPORT=True`) con retornos de estrategia y benchmark sincronizados. Usa `DATA_MODE='csv'`. No mezcles factores sintéticos y reales. Si tienes un CSV de factores, documenta la moneda, frecuencia, fuentes, universo y metodología.

**E8 — Informe final.** ¿De dónde procede la rentabilidad? ¿Qué parte no explica el modelo? ¿Es estable el alpha? ¿Qué factores faltan y qué evidencia necesitarías para operar? No aceptes alpha positivo como evidencia suficiente.

### E1 · Alpha y beta a mano

In [ ]:
print(inspect.getsource(manual_alpha_beta))

e1 = pd.DataFrame([{'strategy': s, 'model': 'manual', **dict(zip(['alpha_bps', 'beta'],
                    (lambda a, b: (a * 1e4, b))(*manual_alpha_beta(train[s] - train.risk_free_return, train.market_excess))))}
                   for s in strategies] +
                  [{'strategy': s, 'model': 'OLS', 'alpha_bps': models[s, 'Mercado'].params['const'] * 1e4,
                    'beta': models[s, 'Mercado'].params['market_excess']} for s in strategies])
display(e1.pivot(index='strategy', columns='model', values=['alpha_bps', 'beta']).round(6))
for s in strategies:
    assert np.allclose(manual_alpha_beta(train[s] - train.risk_free_return, train.market_excess), models[s, 'Mercado'].params.to_numpy())

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · OLS con un factor es covarianza entre varianza</b>

- β = Cov(r_m − r_f, r_s − r_f) / Var(r_m − r_f) = **0,8438** para A y B (comparten exposición).
- α = media(r_s − r_f) − β · media(r_m − r_f) = **6,29 bps** (A) y **10,79 bps** (B).
- Coinciden exactamente con OLS: con un solo factor, la regresión es esa fórmula. Lo que añade `statsmodels` es la **inferencia** (errores HAC, IC y p-valor).
- Se usan **retornos excedentes**, no precios: con precios, la "beta" mediría tendencias comunes de nivel, no la sensibilidad de los retornos.

</div>

### E2 · Retorno activo frente a alpha

In [ ]:
print(inspect.getsource(active_vs_alpha))

display(active_vs_alpha(train, strategies, models, 'Mercado').round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Ganarle al benchmark no es lo mismo que tener alpha</b>

Desarrollo, modelo de mercado:

| | Retorno activo medio (r_s − r_m) | Alpha de Jensen | Beta | Diferencia por la beta |
|---|---|---|---|---|
| A | 6,03 bps/día | 6,29 bps/día | 0,84 | −0,27 bps |
| B | 10,53 bps/día | 10,79 bps/día | 0,84 | −0,27 bps |

- Se cumple exactamente: **media(r_s − r_m) = α + (β − 1) · media(r_m − r_f)**.
- El retorno activo compara con el benchmark **como si la beta fuera 1**. El alpha descuenta la exposición real (β = 0,84). Si el mercado sube, una estrategia con β < 1 queda por detrás en retorno activo aunque tenga alpha positivo. Si cae, le gana sin tener alpha, solo por estar menos expuesta.
- Aquí la diferencia es pequeña (0,27 bps) porque el exceso medio del mercado en desarrollo fue pequeño. En un mercado con una tendencia fuerte, retorno activo y alpha pueden tener incluso signos opuestos.

</div>

### E3 · Factores omitidos

In [ ]:
print(inspect.getsource(omitted_factor_table))

display(pd.concat({s: omitted_factor_table(train, s, factor_cols, HAC_LAGS) for s in strategies}).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Omitir un factor fabrica alpha; un R² alto no valida el modelo</b>

Estrategia A (alpha verdadero = 0):

| Modelo | Alpha diario | IC 95 % | p-valor | Beta | R² |
|---|---|---|---|---|---|
| Completo | 1,09 bps | [−1,23; 3,42] | 0,356 | 0,75 | 0,966 |
| Sin momentum | **5,74 bps** | [0,36; 11,12] | **0,037** | 0,88 | 0,853 |
| Sin liquidez | 1,71 bps | [−1,42; 4,83] | 0,284 | 0,71 | 0,939 |
| Solo mercado | 6,29 bps | [0,44; 12,15] | 0,035 | 0,84 | 0,828 |

- **Quitar momentum fabrica un alpha "significativo"**: la contribución del factor momentum (unos 4,9 bps diarios en A) pasa al intercepto. La beta también cambia (0,75 → 0,88), porque momentum está correlacionado con el mercado.
- Quitar liquidez apenas cambia nada: su prima y su exposición son pequeñas.
- **¿Un R² mayor prueba que el modelo es correcto?** No. El R² mide cuánta varianza explica el modelo, no si el intercepto está bien identificado. Añadir factores siempre sube el R² (aunque sean ruido), un factor puede explicar varianza sin explicar la media, y dos modelos con R² casi iguales pueden dar alphas muy distintos. Lo que da confianza es que los factores tengan una justificación económica previa y que el alpha se mantenga fuera de muestra.

</div>

### E4 · Atribución en porcentaje anual

In [ ]:
print(inspect.getsource(annualized_attribution))

e4 = annualized_attribution(attr_train, PERIODS_PER_YEAR)
display(e4.round(3))
assert np.allclose(e4.total, train[strategies].mean() * PERIODS_PER_YEAR * 100)
print("Suma de componentes = retorno medio diario × 365 (aritmético).")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · De dónde sale cada punto, en % anual aritmético</b>

Desarrollo, modelo multifactor:

| | Libre de riesgo | Alpha (intercepto) | Mercado | Momentum | Liquidez | Residuo | **Total** |
|---|---|---|---|---|---|---|---|
| A | 1,98 % | 3,99 % | 4,69 % | **17,74 %** | 1,84 % | 0,00 % | **30,24 %** |
| B | 1,98 % | **20,42 %** | 4,69 % | 17,74 % | 1,84 % | 0,00 % | **46,67 %** |

- La suma reproduce exactamente el retorno medio diario × 365. Es una suma **aritmética**: no es el CAGR ni el retorno compuesto.
- **Intercepto frente a residuo:** el intercepto (alpha) es la parte **media** que el modelo atribuye a la estrategia y no a los factores. El residuo es lo que sobra **cada día**; en desarrollo su media es exactamente 0, porque OLS lo fuerza. Fuera de muestra, con los parámetros congelados, el residuo medio ya no es 0 (0,19 bps diarios en validación): esa es la parte que el modelo realmente no explica.
- En A, el **momentum** aporta más de la mitad del retorno. En B, la diferencia con A es exactamente el alpha.

</div>

### E5 · Ventanas de 30, 90 y 180 días

In [ ]:
fig, ax = plt.subplots()
e5 = {}
for w, color in [(30, '#b36a37'), (90, '#32658a'), (180, '#26816d')]:
    rb = rolling_beta(daily, strategies[:1], w).iloc[:, 0]
    ax.plot(rb.index, rb, color=color, label=f'{w} días')
    e5[w] = {'desv_tipica': rb.std(), 'minimo': rb.min(), 'maximo': rb.max(), 'primer_valor': rb.first_valid_index().date()}
ax.axvline(test.index[0], color='grey', ls='--'); ax.axhline(.75, color='black', lw=.8, ls=':', label='Beta verdadera (0,75)')
ax.set(title=f'Beta móvil de {strategies[0]} según la ventana', ylabel='Beta'); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
display(pd.DataFrame(e5).T)

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Ventana corta: rápida y ruidosa; ventana larga: estable y lenta</b>

Beta móvil unifactorial de A:

| Ventana | Desv. típica | Mínimo | Máximo | Primer valor |
|---|---|---|---|---|
| 30 días | 0,069 | 0,66 | 1,03 | 30-01-2023 |
| 90 días | 0,031 | 0,78 | 0,92 | 31-03-2023 |
| 180 días | 0,017 | 0,81 | 0,89 | 29-06-2023 |

- La exposición verdadera **es constante**, pero con 30 días la beta estimada se mueve entre 0,66 y 1,03: todo ese movimiento es ruido de estimación. Con 180 días oscila 4 veces menos.
- La beta móvil unifactorial ronda 0,84, no la 0,75 verdadera, porque recoge también la parte del momentum correlacionada con el mercado (como en E3).
- Elegir la ventana mirando el gráfico de validación sería ajustar un parámetro con el test. Si se cambia, hace falta un periodo nuevo para evaluar.

</div>

### E6 · Factor de reversión y costes

In [ ]:
print(inspect.getsource(factor_after_costs))

reversal_ls = long_short_factor(universe, 20, 2, reverse=True)
reversal = reversal_ls['factor']
assert np.allclose((factor + reversal).dropna(), 0)  # es exactamente el momentum con el signo cambiado
print('Primer retorno disponible:', reversal.first_valid_index(), '| rotación media diaria:', round(reversal_ls['turnover'].mean(), 3))
display(pd.DataFrame({cost: {'momentum_bps': factor_after_costs(factor, momentum_ls['turnover'], cost).mean() * 1e4,
                             'reversion_bps': factor_after_costs(reversal, reversal_ls['turnover'], cost).mean() * 1e4}
                      for cost in (0, 10, 20, 50)}).T.rename_axis('coste_por_lado_bps').round(2))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Invertir los grupos invierte el signo, pero no los costes</b>

- El factor de reversión (comprar los 2 peores y vender los 2 mejores) es **exactamente** el momentum con el signo cambiado. Está disponible desde el 21 de enero de 2024, igual que el momentum: necesita 20 días de historia y se ordena con datos **hasta ayer**.
- **Rotación:** de media se cambia un 29 % de la cartera al día (cambia algún activo en el 49 % de los días), y el primer día se construye la cartera completa.

| Coste por lado | Momentum | Reversión |
|---|---|---|
| 0 bps | +10,03 bps/día | −10,03 bps/día |
| 10 bps | +4,18 | −15,87 |
| 20 bps | −1,67 | −21,72 |
| 50 bps | −19,21 | −39,26 |

- Los costes **no** se invierten al invertir los grupos: los dos factores pagan la misma rotación. Con 20 bps por lado, ni siquiera el momentum (que en esta muestra rinde +10 bps diarios en bruto) queda positivo.
- Con activos independientes, ese +10 bps es **azar** de una muestra de 130 días: no hay ninguna prima real que capturar.
- Faltan además el coste de financiar los cortos y su disponibilidad.

</div>

### E7 · Datos propios: una regla real sobre Kraken
No hay un `daily_returns.csv` real de P09 (el de P09 es sintético). Construimos una estrategia diaria real y sencilla: **long/cash en BTC/EUR si el momentum de 20 días al cierre de ayer es positivo**, con 20 bps por ida y vuelta, sobre velas diarias de Kraken. El benchmark es BTC/EUR. No hay factores reales disponibles, así que solo se usa el de mercado: **no se mezclan factores sintéticos con datos reales**. Cambia `LIVE_E7` a `True` para repetirlo.

In [ ]:
print(inspect.getsource(daily_rule_returns))

LIVE_E7 = False

if LIVE_E7:
    from quantlab.p05.signals import fetch_kraken_ohlc, regularize
    btc_close, btc_prov = fetch_kraken_ohlc('XBTEUR', 1440, '1D')  # velas diarias; si falla, lanza el error
    btc_close = regularize(btc_close, '1D')
    rule = daily_rule_returns(btc_close, 20, 20.)
    real = validate_daily_factors(rule[['strategy', 'benchmark_return']].assign(risk_free_return=rf_daily), ['strategy'])
    print(btc_prov['received_at'], '| días:', len(real), real.index.min().date(), '—', real.index.max().date(),
          '| tiempo invertido:', round(rule.position.mean(), 3), '| cambios de posición:', int(rule.position.diff().abs().sum()))
    real_train, real_test = chronological_split(real, TRAIN_FRACTION, HAC_LAGS, 1)
    real_models_table, real_models = model_table(real_train, ['strategy'], {'Mercado': ['market_excess']}, HAC_LAGS, PERIODS_PER_YEAR)
    display(real_models_table.round(4))
    real_validation, _ = frozen_validation(real_test, ['strategy'], real_models, 'Mercado', ['market_excess'], HAC_LAGS, PERIODS_PER_YEAR)
    display(real_validation.round(4))
    real_beta = rolling_beta(real, ['strategy'], ROLLING_WINDOW)
    no_cost = daily_rule_returns(btc_close, 20, 0.)
    print(f"Retorno total neto: {(1 + rule.strategy).prod() - 1:.1%} | sin comisiones: {(1 + no_cost.strategy).prod() - 1:.1%} | "
          f"BTC/EUR: {(1 + rule.benchmark_return).prod() - 1:.1%} | beta móvil de {real_beta.min().iloc[0]:.2f} a {real_beta.max().iloc[0]:.2f}")
    real_beta.plot(title='Regla real · beta móvil de 90 días'); plt.show()
else:
    print("LIVE_E7 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Regla real de momentum sobre BTC/EUR: menos pérdida por menos exposición, no por alpha</b>

**Procedencia:** endpoint OHLC público de Kraken, BTC/EUR, velas diarias (recepción 19-11-2026 21:15 UTC). 720 velas, sin huecos; tras los 20 días de historia, **698 días** de retornos (22-12-2024 a 19-11-2026). Moneda: EUR. Tipo libre de riesgo: supuesto del 2 % anual. La regla estuvo invertida el **43 %** del tiempo y cambió de posición 89 veces.

| | Desarrollo (453 días) | Validación (245 días) |
|---|---|---|
| Retorno de la regla (neto) | −29,3 % | **+2,2 %** |
| Retorno de BTC/EUR | −33,7 % | −7,6 % |
| Alpha diario | **−5,64 bps** (IC [−14,8; 3,6], p = 0,23) | +2,08 bps (diagnóstico, p = 0,78) |
| Beta | 0,27 | — |
| R² | 0,27 | — |

- **¿De dónde viene la rentabilidad?** De estar fuera de BTC parte del tiempo (beta 0,27). En un periodo en que BTC/EUR cayó un 39 %, estar invertido menos de la mitad de los días reduce la pérdida, pero eso es **exposición**, no habilidad.
- **El alpha no es estable ni significativo:** −5,6 bps en desarrollo, +2,1 en validación, con p-valores de 0,23 y 0,78.
- **La beta no es constante:** la beta móvil de 90 días va de 0,11 a 0,82, porque la regla entra y sale del mercado. Una beta única describe mal una estrategia así.
- Sin comisiones, la regla habría perdido un 21,0 % en lugar de un 27,7 %: los costes se comen casi 7 puntos en dos años.

</div>

### E8 · Informe final

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Informe final</b>

**¿De dónde procede la rentabilidad?**
- **A (sintética):** de la exposición al mercado y, sobre todo, al **factor momentum** (17,7 % anual aritmético de 30,2 %). No tiene alpha.
- **B (sintética):** de los mismos factores más un **alpha de 5,6 bps diarios** (20,4 % anual aritmético), estable en desarrollo y validación.
- **Regla real sobre Kraken:** de una **exposición reducida** al mercado (beta 0,27) en un periodo bajista. No hay alpha distinguible de cero.

**¿Qué parte no explica el modelo?** En validación, con los parámetros congelados, el residuo medio es de 0,19 bps diarios para A y B. La parte no explicada de B (5,8 bps) es casi toda alpha. En la regla real, el R² es solo de 0,27: el modelo de un factor explica poco de una estrategia que entra y sale del mercado.

**¿Es estable el alpha?** El de B sí (5,59 → 5,50 bps). El de A desaparece en cuanto se incluyen los factores. El de la regla real cambia de signo.

**¿Qué factores faltan?** En cripto, al menos: tamaño (capitalización), momentum entre activos, liquidez o volumen, volatilidad, funding o basis (P07) y un factor de mercado amplio (no solo BTC). Además, la exposición variable en el tiempo (la beta de la regla va de 0,11 a 0,82), que un modelo estático no recoge.

**¿Qué evidencia haría falta para operar?** No basta un alpha positivo:
1. Factores relevantes definidos **antes** del análisis, con datos reales y documentados (moneda, universo, metodología).
2. Alpha significativo con errores robustos (HAC) y **estable** en un periodo de validación nunca usado.
3. Exposiciones estables, o modeladas si varían.
4. Rentabilidad neta de costes **medidos** (P08), con capacidad suficiente y sin depender de unos pocos días.
5. Un mecanismo económico que explique el alpha, y la comprobación de que no es una prima de riesgo omitida.

Con los datos de este cuaderno, **solo B** (que es sintética y tiene el alpha programado) cumpliría la parte estadística. Ninguna estrategia real ha mostrado alpha.

</div>

## 11. Exportación y conexión con el laboratorio
Activa `EXPORT=True` al principio para guardar datos de entrada, resultados, tabla de atribución, beta móvil y metadatos. Las funciones validadas ya están en `src/quantlab/p12/factors.py`, con sus pruebas en `tests/test_factors.py`.

In [ ]:
if EXPORT:
    out=Path('quantlab_exports')/('p12_'+DATA_MODE+'_'+pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ'))
    out.mkdir(parents=True,exist_ok=True)
    daily.to_csv(out/'daily_inputs.csv');model_results.to_csv(out/'models.csv',index=False)
    attr_train.to_csv(out/'attribution_train_bps.csv');validation.to_csv(out/'validation.csv',index=False)
    rolling.to_csv(out/'rolling_beta.csv')
    for s,p in out_parts.items(): p.to_csv(out/f'attribution_test_{s}.csv')
    (out/'metadata.json').write_text(json.dumps({**provenance,'periods_per_year':PERIODS_PER_YEAR,'risk_free_annual_assumption':RISK_FREE_ANNUAL,'train_fraction':TRAIN_FRACTION,'test_start':str(test.index[0]),'factors':factor_cols,'hac_lags':HAC_LAGS,'rolling_window':ROLLING_WINDOW},indent=2),encoding='utf-8')
    print(out)
else: print('Exportación desactivada.')

## 12. Conclusiones y límites
Un alpha estimado no es necesariamente una ventaja causal ni explotable. La evaluación debe ser neta de costes, con exposición realista, factores pertinentes y un periodo reservado que no se utilice para seleccionar modelos. Los factores de este cuaderno son artificiales salvo que importes datos propios; no representan primas empíricas de cripto.

**Referencia de la ejecución sintética (seed 42):** A pasa de 6,29 bps de alpha diario (p=0,0352) con el modelo de mercado a 1,09 bps (p=0,3558) con los tres factores; B conserva 5,59 bps en desarrollo y 5,50 bps en la regresión diagnóstica de validación. Ambas estrategias pierden capital en validación.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P12</b>

- **Un alpha con el modelo equivocado puede ser exposición disfrazada:** A tenía un alpha "significativo" de 6,29 bps diarios que desapareció al incluir el factor momentum.
- **Un alpha real puede convivir con pérdidas:** B mantuvo su alpha fuera de muestra y aun así perdió un 16 % porque el mercado cayó y su beta era de 0,75.
- **Retorno activo ≠ alpha:** la diferencia es (β − 1) por el exceso del mercado.
- **Un R² alto no valida el modelo:** lo que importa es si el intercepto se mantiene con los factores correctos y fuera de muestra.
- **Las exposiciones cambian:** una estrategia que entra y sale del mercado (la regla real) tiene una beta entre 0,11 y 0,82; una beta única la describe mal.
- **Con datos reales, no hay alpha:** la regla de momentum sobre BTC/EUR perdió menos que BTC por estar menos expuesta, no por habilidad.

Con P12 se cierra el bloque principal de fundamentos; P13–P15 lo amplían con order flow, análisis multiactivo y regímenes macro. Antes de pasar a Microestructura avanzada, realiza la evaluación de competencias y una investigación reproducible de principio a fin.

📄 Resumen en `docs/conclusiones/P12_factor_analysis.md`

</div>